## Problem 01: Campus Student Registry & Class-Level State Engine

This problem demonstrates:
- Class variables and instance variables
- Class methods
- Static methods
- Alternative constructors
- Attribute shadowing
- CGPA validation

`university` and `student_count` are class-level attributes, while
`name`, `roll`, and `cgpa` belong to individual student objects.

In [1]:
class Student:
    university = "DS_University"
    student_count = 0

    def __init__(self, name, roll, cgpa):
        if not Student.is_valid_cgpa(cgpa):
            raise ValueError("Invalid CGPA")

        self.name = name
        self.roll = roll
        self.cgpa = cgpa
        Student.student_count += 1

    @classmethod
    def from_string(cls, text):
        name, roll, cgpa = text.split("-")
        return cls(name, roll, float(cgpa))

    @classmethod
    def from_dict(cls, data):
        return cls(
            data["name"],
            data["roll"],
            float(data["cgpa"])
        )

    @classmethod
    def change_university(cls, new_name):
        cls.university = new_name

    @staticmethod
    def is_valid_cgpa(cgpa):
        return 0.0 <= cgpa <= 10.0

    @staticmethod
    def letter_grade(cgpa):
        if cgpa >= 9:
            return "A"
        elif cgpa >= 8:
            return "B"
        elif cgpa >= 7:
            return "C"
        elif cgpa >= 6:
            return "D"
        else:
            return "F"

    def __str__(self):
        return (
            f"{self.name} ({self.roll}) | "
            f"CGPA: {self.cgpa:.2f} | "
            f"Grade: {self.letter_grade(self.cgpa)} | "
            f"University: {self.university}"
        )


students = {}

Q = int(input())

for _ in range(Q):
    command = input().split()

    if command[0] == "ADD":
        name = command[1]
        roll = command[2]
        cgpa = float(command[3])

        try:
            student = Student(name, roll, cgpa)
            students[roll] = student
            print(f"Student Added: {name} ({roll})")
        except ValueError:
            print("Error: Invalid CGPA")

    elif command[0] == "ADD_STR":
        try:
            student = Student.from_string(command[1])
            students[student.roll] = student
            print(f"Student Added: {student.name} ({student.roll})")
        except ValueError:
            print("Error: Invalid CGPA")

    elif command[0] == "ADD_DICT":
        data = {}

        for item in command[1:]:
            key, value = item.split("=")
            data[key] = value

        try:
            student = Student.from_dict(data)
            students[student.roll] = student
            print(f"Student Added: {student.name} ({student.roll})")
        except ValueError:
            print("Error: Invalid CGPA")

    elif command[0] == "RENAME_UNI":
        Student.change_university(command[1])
        print(f"University set to: {command[1]}")

    elif command[0] == "SHADOW":
        roll = command[1]
        university = command[2]

        students[roll].university = university
        print(f"Instance override set for {roll}")

    elif command[0] == "INFO":
        roll = command[1]
        print(students[roll])

    elif command[0] == "COUNT":
        print(f"Total Students: {Student.student_count}")

    elif command[0] == "GRADE":
        cgpa = float(command[1])
        print(f"Grade for {command[1]}: {Student.letter_grade(cgpa)}")

Student Added: ASHA (R1)
Student Added: Ravi (R2)
Error: Invalid CGPA
Student Added: Kiran (R4)
University set to: Global_Tech
Instance override set for R2
ASHA (R1) | CGPA: 8.50 | Grade: B | University: Global_Tech
Ravi (R2) | CGPA: 9.20 | Grade: A | University: Legacy_Institute
Total Students: 3
Grade for 7.5: C


## Problem 02: Corporate Payroll Hierarchy

This problem demonstrates:
- Multilevel inheritance
- Hierarchical inheritance
- Method overriding
- `super()`
- Polymorphism
- `isinstance()`
- MRO

All employee objects can be stored in one list and their
`monthly_pay()` method can be called polymorphically.

In [5]:
class Employee:
    role = "Employee"

    def __init__(self, name, base_salary):
        self.name = name
        self.base_salary = float(base_salary)

    def monthly_pay(self):
        return self.base_salary

    def _extra(self):
        return ""

    def describe(self):
        return f"{self.name} [{self.role}{self._extra()}]"


class Manager(Employee):
    role = "Manager"

    def __init__(self, name, base_salary, team_size):
        super().__init__(name, base_salary)
        self.team_size = int(team_size)

    def monthly_pay(self):
        return super().monthly_pay() + 500 * self.team_size

    def _extra(self):
        return f", team={self.team_size}"


class Director(Manager):
    role = "Director"

    def monthly_pay(self):
        return super().monthly_pay() * 1.20


class Engineer(Employee):
    role = "Engineer"

    def __init__(self, name, base_salary, level):
        super().__init__(name, base_salary)
        self.level = int(level)

    def monthly_pay(self):
        return super().monthly_pay() * (1 + 0.10 * self.level)

    def _extra(self):
        return f", level={self.level}"


class Intern(Employee):
    role = "Intern"

    def monthly_pay(self):
        return super().monthly_pay() * 0.5


employees = {}
employee_list = []

N = int(input())

for _ in range(N):
    command = input().split()

    employee_type = command[0]
    name = command[1]
    base_salary = float(command[2])

    if employee_type == "EMPLOYEE":
        employee = Employee(name, base_salary)

    elif employee_type == "MANAGER":
        team_size = int(command[3])
        employee = Manager(name, base_salary, team_size)

    elif employee_type == "DIRECTOR":
        team_size = int(command[3])
        employee = Director(name, base_salary, team_size)

    elif employee_type == "ENGINEER":
        level = int(command[3])
        employee = Engineer(name, base_salary, level)

    elif employee_type == "INTERN":
        employee = Intern(name, base_salary)

    employees[name] = employee
    employee_list.append(employee)


total = 0
highest = employee_list[0]

for employee in employee_list:
    pay = employee.monthly_pay()
    total += pay

    if pay > highest.monthly_pay():
        highest = employee

    print(
        f"{employee.describe()} -> "
        f"Pay: ${pay:.2f}"
    )

print(f"Total Monthly Payroll: ${total:.2f}")

highest_pay = highest.monthly_pay()
print(
    f"Highest Paid: {highest.name} "
    f"(${highest_pay:.2f})"
)


Q = int(input())

for _ in range(Q):
    command = input().split()

    if command[0] == "ISA":
        name = command[1]
        class_name = command[2]

        employee = employees[name]

        class_map = {
            "Employee": Employee,
            "Manager": Manager,
            "Director": Director,
            "Engineer": Engineer,
            "Intern": Intern
        }

        print(isinstance(employee, class_map[class_name]))

    elif command[0] == "MRO":
        class_name = command[1]

        class_map = {
            "Employee": Employee,
            "Manager": Manager,
            "Director": Director,
            "Engineer": Engineer,
            "Intern": Intern
        }

        cls = class_map[class_name]

        print(
            " -> ".join(
                c.__name__ for c in cls.__mro__
            )
        )

Anil [Employee] -> Pay: $3000.00
Bhavna [Manager, team=4] -> Pay: $7000.00
Chetan [Director, team=10] -> Pay: $15600.00
Divya [Engineer, level=3] -> Pay: $5200.00
Esha [Intern] -> Pay: $1000.00
Total Monthly Payroll: $31800.00
Highest Paid: Chetan ($15600.00)
True
False
True
Director -> Manager -> Employee -> object


## Problem 03: Smart-Home Device Fabric

This problem demonstrates:
- Multiple inheritance
- Diamond inheritance
- C3 Method Resolution Order
- Cooperative `super()`
- MRO introspection

Every `power_on()` method calls `super().power_on()` first.
Therefore, the shared `Device` boot operation occurs exactly once.

In [6]:
class Device:
    feature = None

    def __init__(self, name):
        self.name = name
        self.log = []

    def power_on(self):
        self.log = ["Device core booted"]

    def mro_names(self):
        return " -> ".join(
            cls.__name__
            for cls in type(self).__mro__
        )

    def capabilities(self):
        return [
            cls.feature
            for cls in type(self).__mro__
            if vars(cls).get("feature") is not None
        ]


class Camera(Device):
    feature = "video"

    def power_on(self):
        super().power_on()
        self.log.append("Camera sensor online")


class Speaker(Device):
    feature = "audio"

    def power_on(self):
        super().power_on()
        self.log.append("Speaker amplifier online")


class Sensor(Device):
    feature = "motion"

    def power_on(self):
        super().power_on()
        self.log.append("Motion sensor armed")


class Doorbell(Camera, Speaker):
    feature = "chime"

    def power_on(self):
        super().power_on()
        self.log.append("Doorbell chime ready")


class SecurityHub(Doorbell, Sensor):
    feature = "uplink"

    def power_on(self):
        super().power_on()
        self.log.append("Hub uplink established")


devices = {}

Q = int(input())

for _ in range(Q):
    command = input().split()

    if command[0] == "NEW":
        device_type = command[1]
        name = command[2]

        class_map = {
            "CAMERA": Camera,
            "SPEAKER": Speaker,
            "SENSOR": Sensor,
            "DOORBELL": Doorbell,
            "HUB": SecurityHub
        }

        if device_type not in class_map:
            print("Error: Unknown Device Type")
            continue

        device = class_map[device_type](name)
        devices[name] = device

        print(
            f"Created {type(device).__name__} "
            f"'{name}'"
        )

    elif command[0] == "POWER_ON":
        name = command[1]

        if name not in devices:
            print("Error: Unknown Device")
            continue

        device = devices[name]
        device.power_on()

        print(f"Boot sequence for {name}:")

        for i, message in enumerate(device.log, 1):
            print(f"  {i}. {message}")

    elif command[0] == "MRO":
        name = command[1]

        if name not in devices:
            print("Error: Unknown Device")
            continue

        print(
            f"MRO({name}): "
            f"{devices[name].mro_names()}"
        )

    elif command[0] == "CAPS":
        name = command[1]

        if name not in devices:
            print("Error: Unknown Device")
            continue

        capabilities = devices[name].capabilities()

        print(
            f"Capabilities({name}): "
            f"{', '.join(capabilities)}"
        )

Created Doorbell 'Front'
Created SecurityHub 'Home'
Error: Unknown Device Type
Boot sequence for Front:
  1. Device core booted
  2. Speaker amplifier online
  3. Camera sensor online
  4. Doorbell chime ready
Boot sequence for Home:
  1. Device core booted
  2. Motion sensor armed
  3. Speaker amplifier online
  4. Camera sensor online
  5. Doorbell chime ready
  6. Hub uplink established
MRO(Home): SecurityHub -> Doorbell -> Camera -> Speaker -> Sensor -> Device -> object
Capabilities(Home): uplink, chime, video, audio, motion
Error: Unknown Device


## Problem 04: Smart Shopping Cart

This problem demonstrates:
- Composition
- HAS-A relationship
- `__len__`
- `__getitem__`
- `__contains__`
- `__iter__`
- `__bool__`
- `__call__`
- `__repr__`
- `__eq__`
- `__hash__`
- Simulated method overloading using `*args`

The Cart contains Item objects instead of inheriting from `list`.

In [7]:
class Item:
    def __init__(self, name, price, qty=1):
        self.name = name
        self.price = float(price)
        self.qty = int(qty)

    @property
    def subtotal(self):
        return self.price * self.qty

    def __eq__(self, other):
        if not isinstance(other, Item):
            return NotImplemented

        return (
            self.name == other.name
            and self.price == other.price
        )

    def __hash__(self):
        return hash((self.name, self.price))

    def __repr__(self):
        return (
            f"Item('{self.name}', "
            f"{self.price}, {self.qty})"
        )


class Cart:
    def __init__(self, owner):
        self.owner = owner
        self._items = []

    def add(self, *args):
        if len(args) == 1 and isinstance(args[0], Item):
            item = args[0]

        elif len(args) == 2:
            item = Item(
                args[0],
                args[1],
                1
            )

        elif len(args) == 3:
            item = Item(
                args[0],
                args[1],
                args[2]
            )

        else:
            raise TypeError("Unsupported arguments")

        for existing in self._items:
            if existing.name == item.name:
                existing.qty += item.qty
                return

        self._items.append(item)

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return bool(self._items)

    def __contains__(self, name):
        return any(
            item.name == name
            for item in self._items
        )

    def __iter__(self):
        return iter(self._items)

    def __getitem__(self, key):
        if isinstance(key, int):
            if key < 0 or key >= len(self._items):
                raise IndexError("Index out of range")

            return self._items[key]

        if isinstance(key, str):
            for item in self._items:
                if item.name == key:
                    return item

            raise KeyError(key)

        raise TypeError("Invalid key")

    def __call__(self, discount_percent=0):
        total = sum(
            item.subtotal
            for item in self._items
        )

        return total * (
            1 - discount_percent / 100
        )

    def __repr__(self):
        return (
            f"Cart(owner='{self.owner}', "
            f"items={len(self)})"
        )

    def remove(self, name):
        for i, item in enumerate(self._items):
            if item.name == name:
                del self._items[i]
                return

        raise KeyError(name)


owner = input()
cart = Cart(owner)

Q = int(input())

for _ in range(Q):
    command = input().split()

    if command[0] == "ADD":
        name = command[1]
        price = float(command[2])

        if len(command) == 4:
            qty = int(command[3])
            cart.add(name, price, qty)
        else:
            cart.add(name, price)

        print(f"Added: {name}")

    elif command[0] == "ADD_ITEM":
        name = command[1]
        price = float(command[2])
        qty = int(command[3])

        item = Item(name, price, qty)
        cart.add(item)

        print(f"Added: {name}")

    elif command[0] == "REMOVE":
        name = command[1]

        try:
            cart.remove(name)
            print(f"Removed: {name}")
        except KeyError:
            print("Error: Item Not Found")

    elif command[0] == "GET":
        key = command[1]

        try:
            if key.isdigit():
                item = cart[int(key)]
            else:
                item = cart[key]

            print(repr(item))

        except IndexError:
            print("Error: Index Out of Range")

        except KeyError:
            print("Error: Item Not Found")

    elif command[0] == "HAS":
        name = command[1]
        print(name in cart)

    elif command[0] == "LEN":
        print(f"Distinct Items: {len(cart)}")

    elif command[0] == "EMPTY":
        print(f"Cart Empty: {not bool(cart)}")

    elif command[0] == "LIST":
        for i, item in enumerate(cart, 1):
            print(
                f"{i}. {item.name} "
                f"x{item.qty} @ "
                f"${item.price:.2f} = "
                f"${item.subtotal:.2f}"
            )

    elif command[0] == "TOTAL":
        pct = float(command[1])

        total = cart(pct)

        print(
            f"Total ({pct:g}% off): "
            f"${total:.2f}"
        )

    elif command[0] == "REPR":
        print(repr(cart))

Cart Empty: True
Added: pen
Added: notebook
Added: bag
Added: pen
1. pen x5 @ $2.50 = $12.50
2. notebook x2 @ $45.00 = $90.00
3. bag x1 @ $999.99 = $999.99
Item('notebook', 45.0, 2)
Item('bag', 999.99, 1)
Error: Index Out of Range
False
Distinct Items: 3
Removed: notebook
Total (10% off): $911.24
Cart(owner='Zara', items=2)


## Problem 05: Mini Core-Banking Platform

This problem combines:
- Abstract Base Classes
- Encapsulation
- Inheritance
- Method overriding
- Composition
- Class variables
- Class methods
- Static methods
- Factory design
- Polymorphism
- Atomic transfers

`Account` protects its balance using a private attribute.
Concrete account types implement their own `available_funds()`
and `monthly_update()` behavior.

In [8]:
from abc import ABC, abstractmethod


class Account(ABC):
    kind = None

    def __init__(self, account_id, balance):
        self.account_id = account_id
        self.__balance = float(balance)

    @property
    def balance(self):
        return self.__balance

    def _apply(self, delta):
        self.__balance += delta

    @abstractmethod
    def available_funds(self):
        pass

    @abstractmethod
    def monthly_update(self):
        pass


class SavingsAccount(Account):
    kind = "Savings"
    MONTHLY_RATE = 0.01

    def available_funds(self):
        return self.balance

    def monthly_update(self):
        interest = self.balance * self.MONTHLY_RATE
        self._apply(interest)

        return (
            f"{self.account_id} (Savings): "
            f"+${interest:.2f} interest -> "
            f"${self.balance:.2f}"
        )


class CurrentAccount(Account):
    kind = "Current"
    OVERDRAFT_LIMIT = 500.0
    MONTHLY_FEE = 10.0

    def available_funds(self):
        return self.balance + self.OVERDRAFT_LIMIT

    def monthly_update(self):
        self._apply(-self.MONTHLY_FEE)

        return (
            f"{self.account_id} (Current): "
            f"-${self.MONTHLY_FEE:.2f} fee -> "
            f"${self.balance:.2f}"
        )


class Customer:
    def __init__(self, customer_id, name):
        self.customer_id = customer_id
        self.name = name
        self.accounts = {}


class Bank:
    bank_name = ""
    total_transactions = 0

    ACCOUNT_TYPES = {
        "SAVINGS": SavingsAccount,
        "CURRENT": CurrentAccount
    }

    def __init__(self, name):
        Bank.bank_name = name
        self.customers = {}
        self.accounts = {}

    @classmethod
    def rename(cls, new_name):
        cls.bank_name = new_name

    @staticmethod
    def valid_amount(x):
        return x > 0

    def add_customer(self, customer):
        self.customers[customer.customer_id] = customer

    def open_account(
        self,
        customer_id,
        account_id,
        kind,
        initial
    ):
        customer = self.customers[customer_id]

        account_class = self.ACCOUNT_TYPES[kind]

        account = account_class(
            account_id,
            initial
        )

        customer.accounts[account_id] = account
        self.accounts[account_id] = account

        return account

    def deposit(self, account_id, amount):
        if account_id not in self.accounts:
            raise KeyError("Account Not Found")

        if not self.valid_amount(amount):
            raise ValueError("Invalid Amount")

        account = self.accounts[account_id]

        account._apply(amount)
        Bank.total_transactions += 1

    def withdraw(self, account_id, amount):
        if account_id not in self.accounts:
            raise KeyError("Account Not Found")

        if not self.valid_amount(amount):
            raise ValueError("Invalid Amount")

        account = self.accounts[account_id]

        if amount > account.available_funds():
            raise RuntimeError("Insufficient Funds")

        account._apply(-amount)
        Bank.total_transactions += 1

    def transfer(
        self,
        from_account,
        to_account,
        amount
    ):
        if (
            from_account not in self.accounts
            or to_account not in self.accounts
        ):
            raise KeyError("Account Not Found")

        if not self.valid_amount(amount):
            raise ValueError("Invalid Amount")

        source = self.accounts[from_account]
        destination = self.accounts[to_account]

        if amount > source.available_funds():
            raise RuntimeError("Insufficient Funds")

        source._apply(-amount)
        destination._apply(amount)

        Bank.total_transactions += 1

    def month_end(self):
        for account in self.accounts.values():
            print(account.monthly_update())


bank_name = input()
bank = Bank(bank_name)

K = int(input())

for _ in range(K):
    customer_id, name = input().split()

    customer = Customer(
        customer_id,
        name
    )

    bank.add_customer(customer)


Q = int(input())

for _ in range(Q):
    command = input().split()

    try:

        if command[0] == "OPEN":
            cid = command[1]
            accid = command[2]
            kind = command[3]
            initial = float(command[4])

            account = bank.open_account(
                cid,
                accid,
                kind,
                initial
            )

            customer = bank.customers[cid]

            print(
                f"Opened {account.kind} account "
                f"{accid} for {customer.name} "
                f"with ${initial:.2f}"
            )

        elif command[0] == "DEPOSIT":
            accid = command[1]
            amount = float(command[2])

            bank.deposit(accid, amount)

            balance = bank.accounts[accid].balance

            print(
                f"Deposited ${amount:.2f} to {accid}. "
                f"Balance: ${balance:.2f}"
            )

        elif command[0] == "WITHDRAW":
            accid = command[1]
            amount = float(command[2])

            bank.withdraw(accid, amount)

            balance = bank.accounts[accid].balance

            if balance < 0:
                balance_text = f"-${abs(balance):.2f}"
            else:
                balance_text = f"${balance:.2f}"

            print(
                f"Withdrew ${amount:.2f} from "
                f"{accid}. Balance: {balance_text}"
            )

        elif command[0] == "TRANSFER":
            source = command[1]
            destination = command[2]
            amount = float(command[3])

            bank.transfer(
                source,
                destination,
                amount
            )

            print(
                f"Transferred ${amount:.2f} "
                f"from {source} to {destination}"
            )

        elif command[0] == "MONTH_END":
            bank.month_end()

        elif command[0] == "RENAME":
            new_name = command[1]

            Bank.rename(new_name)

            print(
                f"Bank renamed to: {new_name}"
            )

        elif command[0] == "SUMMARY":
            cid = command[1]

            customer = bank.customers[cid]

            print(
                f"Customer {customer.name} "
                f"({cid}): "
                f"{len(customer.accounts)} account(s)"
            )

            net_worth = 0

            for account in customer.accounts.values():
                balance = account.balance
                net_worth += balance

                if balance < 0:
                    balance_text = f"-${abs(balance):.2f}"
                else:
                    balance_text = f"${balance:.2f}"

                print(
                    f"{account.account_id} "
                    f"{account.kind}: "
                    f"{balance_text}"
                )

            if net_worth < 0:
                net_text = f"-${abs(net_worth):.2f}"
            else:
                net_text = f"${net_worth:.2f}"

            print(f"Net Worth: {net_text}")

        elif command[0] == "STATS":
            print(
                f"Bank: {Bank.bank_name} | "
                f"Transactions: "
                f"{Bank.total_transactions}"
            )

    except KeyError:
        print("Error: Account Not Found")

    except ValueError:
        print("Error: Invalid Amount")

    except RuntimeError:
        print("Error: Insufficient Funds")

Opened Savings account S1 for Alice with $1000.00
Opened Current account K1 for Alice with $200.00
Opened Savings account S2 for Bob with $500.00
Deposited $250.00 to S1. Balance: $1250.00
Withdrew $600.00 from K1. Balance: -$400.00
Error: Insufficient Funds
Transferred $300.00 from S1 to K1
Error: Insufficient Funds
S1 (Savings): +$9.50 interest -> $959.50
K1 (Current): -$10.00 fee -> $-110.00
S2 (Savings): +$5.00 interest -> $505.00
Bank renamed to: NovaTrust
Customer Alice (C01): 2 account(s)
S1 Savings: $959.50
K1 Current: -$110.00
Net Worth: $849.50
Bank: NovaTrust | Transactions: 3
Error: Account Not Found
